# Exploring the plant-health benchmark

Welcome! This notebook is a guided tour of the benchmark: the dataset, the
feature representations, the models, and the results. Each section runs on
its own in a few seconds, so you can read top to bottom and experiment as
you go.

**What you will see**
1. The three data files behind each crop
2. How a feature representation reshapes the data
3. How a plant's visit history becomes a sequence
4. Training a model on a small slice of the benchmark
5. The full shipped results: rankings, win/loss tables, and figures


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "code"))
from pipeline import Cell, DRO, MODELS

pd.set_option("display.width", 120)
print("ready")

## 1. The dataset

Every crop ships with three files, collected over one growing season:

* **Field observations** - each plant visited roughly weekly, with disease
  scores and plant traits recorded on every visit.
* **Daily weather** - station measurements enriched with agronomic
  variables (rolling aggregates, degree-days, leaf-wetness hours).
* **Plant distances** - pairwise distances between the monitored plants of
  each farm, enabling spatial context.

`Cell` assembles all three into one modeling table: one row is one plant
on one sampling day.

In [ ]:
for crop in ("Carrot", "Lettuce", "Onion"):
    cell = Cell(crop, "classification")
    visits = pd.Series([len(p) + 1 for p in cell.past])
    print(f"{crop:8s} rows={cell.n:4d}  features={cell.data.X.shape[1]:3d}  "
          f"plants={len(set(zip(cell.farm, cell.plant))):3d}  "
          f"visits/plant median={int(visits.median())}  "
          f"positive rate={cell.y.mean():.2f}")

## 2. Feature representations

The benchmark evaluates six representations of the sampling-day features.
`Cell.project` fits the chosen reduction on training rows and projects the
whole table - here is how the dimensionality changes, and what the leading
PCA plane looks like for lettuce.

In [ ]:
cell = Cell("Lettuce", "classification")
train_idx = np.arange(0, 430)

for dr in DRO:
    Z = cell.project(train_idx, dr, seed=0)
    print(f"{dr:13s} -> {Z.shape[1]:3d} dimensions")

Z = cell.project(train_idx, "PCA", seed=0)
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=cell.sev, cmap="RdYlGn_r", s=14)
plt.colorbar(sc, label="disease severity")
plt.xlabel("PCA component 1"); plt.ylabel("PCA component 2")
plt.title("Lettuce sampling days on the leading PCA plane")
plt.tight_layout(); plt.show()

## 3. From visits to sequences

The sequence models read each plant's recent history: every step carries
the projected features of one past visit together with the disease context
observed on that visit, the neighbor severity of its sweep, and the time
gap. The final step is the sampling day itself, carrying the most recent
observed context. Everything is strictly backward-looking.

In [ ]:
K = 4
S = cell.sequences(np.array([300]), Z, K)
print("sequence tensor:", S.shape, "(1 plant-day, K+1 steps, features+context)")
ctx = S[0, :, -8:]
frame = pd.DataFrame(ctx, columns=[
    "severity", "other disease 1", "other disease 2", "neighbor severity",
    "days before", "step marker", "observed", "days since last"])
frame.index = [f"visit -{K - i}" for i in range(K)] + ["sampling day"]
frame.round(3)

## 4. Train a model on a small slice

The full benchmark runs from the command line (`python run_all.py`), and
every piece is importable. Here we train one gradient-based sequence model
on a single crop and representation - a miniature version of one benchmark
cell.

In [ ]:
from nets import make_net, train_net, predict_net
from dr_agri.evaluate import _outer_split
from sklearn.metrics import f1_score

cellc = Cell("Carrot", "classification")
tr, te = _outer_split(cellc.y, "classification", seed=0)
tr_fit, va = tr[:240], tr[240:]
Zc = cellc.project(tr_fit, "PCA", seed=0)
X_tr = cellc.sequences(tr_fit, Zc, K=4)
X_va = cellc.sequences(va, Zc, K=4)
X_te = cellc.sequences(te, Zc, K=4)

net = make_net("RNN GRU", "classification", dict(K=4, units=32, seed=0),
               X_tr.shape[-1])
net, epochs = train_net(net, "classification", X_tr, cellc.y[tr_fit],
                        X_va, cellc.y[va], seed=0)
pred = predict_net(net, "classification", X_te)
print(f"GRU on Carrot/PCA: F1 = {f1_score(cellc.y[te], pred):.4f} "
      f"({epochs} epochs)")

## 5. The shipped results

`results/runs.csv` holds the complete benchmark: 2,880 runs with scores,
timings, and the configuration selected for every run. The commands below
reproduce the headline outputs; `outputs.py` writes all of them to
`outputs/` as CSV tables and PNG figures.

In [ ]:
runs = pd.read_csv("results/runs.csv")
cells_df = runs.groupby(["task", "crop", "dr", "model"], as_index=False).agg(
    m=("score", "mean"))
for task in ("classification", "regression"):
    r = (cells_df[cells_df.task == task].groupby("model").m.mean()
         .sort_values(ascending=False))
    print(f"\n{task} - overall ranking")
    for rank, (m, v) in enumerate(r.items(), start=1):
        print(f"  {rank}. {m:14s} {v:.4f}")

In [ ]:
from outputs import win_loss_tables, figures
out_tables = Path("outputs/tables"); out_tables.mkdir(parents=True, exist_ok=True)
out_figs = Path("outputs/figures"); out_figs.mkdir(parents=True, exist_ok=True)
win_loss_tables(runs, out_tables)
pd.read_csv(out_tables / "win_loss_regression.csv")

In [ ]:
figures(runs, out_figs)
from IPython.display import Image
Image(str(out_figs / "Figure_5.png"), width=560)

## 6. Compare with the published baseline

`published_baseline.csv` carries the originally published per-cell scores
of the six classical and neural models, enabling a direct cell-by-cell
comparison with any run you produce.

In [ ]:
pub = pd.read_csv("data/published_baseline.csv")
new = runs.groupby(["task", "crop", "dr", "model"], as_index=False).agg(
    new=("score", "mean"))
cmp = pub.merge(new, on=["task", "crop", "dr", "model"])
cmp["delta"] = (cmp.new - cmp.published).round(4)
cmp.groupby(["task", "model"]).delta.mean().round(4).unstack("task")

## Where to go next

* `python run_all.py --help` lists every parameter - any subset of tasks,
  crops, representations, models, and seeds can be run on its own.
* `outputs.py` regenerates every table and figure from any results
  directory.
* `report.py` prints the rankings and win/loss tables as markdown.

Enjoy exploring!